# S1.4 — CI/CD Setup Verification

This notebook validates the GitHub Actions CI workflow created for PaperAlchemy.

**Spec**: `specs/spec-S1.4-ci-cd-setup/`
**Target**: `.github/workflows/ci.yml`
**Jobs**: Lint (Ruff), Type Check (mypy), Test (pytest+coverage), Docker Build

In [ ]:
from pathlib import Path
import yaml

PROJECT_ROOT = Path.cwd().parent.parent
CI_WORKFLOW = PROJECT_ROOT / ".github" / "workflows" / "ci.yml"

assert CI_WORKFLOW.exists(), f"Missing: {CI_WORKFLOW}"
workflow = yaml.safe_load(CI_WORKFLOW.read_text())
print(f"✅ Workflow file exists: {CI_WORKFLOW.relative_to(PROJECT_ROOT)}")
print(f"   Name: {workflow['name']}")

## Trigger Configuration

In [ ]:
triggers = workflow.get("on") or workflow.get(True, {})

push_branches = triggers["push"]["branches"]
pr_branches = triggers["pull_request"]["branches"]

print(f"✅ Push triggers on: {push_branches}")
print(f"✅ PR triggers on: {pr_branches}")
print(f"   Paths ignored: {triggers['push'].get('paths-ignore', 'none')}")

## Job Inventory

In [ ]:
jobs = workflow["jobs"]

print(f"Total jobs: {len(jobs)}\n")
for name, config in jobs.items():
    needs = config.get("needs", [])
    runner = config.get("runs-on", "?")
    steps = config.get("steps", [])
    step_names = [s.get("name", "unnamed") for s in steps]
    deps = f" (needs: {needs})" if needs else " (independent)"
    print(f"📋 {name}{deps}")
    print(f"   Runner: {runner}")
    print(f"   Steps ({len(steps)}): {', '.join(step_names)}")
    print()

## Validate Key Requirements

In [ ]:
checks = []

# 1. Lint job has ruff check + format
lint_steps = yaml.dump(jobs["lint"]["steps"])
checks.append(("Lint: ruff check", "ruff check" in lint_steps))
checks.append(("Lint: ruff format --check", "ruff format" in lint_steps))

# 2. Type-check job has mypy
tc_steps = yaml.dump(jobs["type-check"]["steps"])
checks.append(("Type-check: mypy", "mypy" in tc_steps))

# 3. Test job has pytest + coverage
test_steps = yaml.dump(jobs["test"]["steps"])
checks.append(("Test: pytest", "pytest" in test_steps))
checks.append(("Test: coverage", "cov" in test_steps))

# 4. Docker build job exists
docker_steps = yaml.dump(jobs["docker-build"]["steps"])
checks.append(("Docker: buildx", "buildx" in docker_steps.lower() or "docker" in docker_steps.lower()))
checks.append(("Docker: no push", "push: false" in docker_steps))

# 5. Python 3.12 used
for job_name in ["lint", "type-check", "test"]:
    job_str = yaml.dump(jobs[job_name]["steps"])
    checks.append((f"{job_name}: Python 3.12", "3.12" in job_str))

# 6. UV caching
for job_name in ["lint", "type-check", "test"]:
    job_str = yaml.dump(jobs[job_name]["steps"])
    checks.append((f"{job_name}: UV cache", "enable-cache" in job_str))

# 7. Parallel execution (lint, type-check, test have no inter-deps)
parallel_jobs = ["lint", "type-check", "test"]
for jn in parallel_jobs:
    needs = jobs[jn].get("needs", [])
    checks.append((f"{jn}: independent (parallel)", len(needs) == 0 if isinstance(needs, list) else False))

# 8. Docker build depends on test
docker_needs = jobs["docker-build"].get("needs", [])
checks.append(("Docker: needs test", "test" in docker_needs))

print("Requirement Validation:\n")
all_pass = True
for label, passed in checks:
    status = "✅" if passed else "❌"
    if not passed:
        all_pass = False
    print(f"  {status} {label}")

print(f"\n{'✅ ALL CHECKS PASSED' if all_pass else '❌ SOME CHECKS FAILED'}")

## Run pytest (S1.4 tests)

In [ ]:
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "tests/unit/test_ci_cd_setup.py", "-v", "--tb=short"],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT),
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)